# Wastewater 16S Pathogenicity Prediction — Genome-Resolved Pipeline

**GOMICS 16S track · GEAR Center / Digital Twin Systems Lab · University of South Dakota**

This notebook runs the full 16S pathogen-prediction pipeline end to end on an HPC
cluster, with **one cell per objective**, mirroring the whole-genome MAG repository
[`bicbioeng/wastewater-mag-pathogenicity-prediction`](https://github.com/bicbioeng/wastewater-mag-pathogenicity-prediction)
and adapting each step to the single-gene 16S setting.

The 16S method is a **two-model ensemble**:

| | Question | Method |
|---|---|---|
| **Model A** — identify | *Which organism is this?* | BLAST nearest-neighbour vs a 26,877-sequence 16S RefSeq reference then taxonomy + rank |
| **Model B** — classify | *Is it pathogenic?* | canonical 6-mer vector then class-balanced Random Forest then P(pathogen) |
| **Fusion** — decide | *One honest call* | precision-biased rule layer with a reject option: PATHOGEN / OPPORTUNIST / NON-PATHOGEN / REVIEW / UNKNOWN |

**Objectives (one cell each):**
0. Environment & config
1. Build the Model A reference (16S RefSeq then BLAST DB)
2. Build the Model B labeled set (826 pathogen + 606 non-pathogen 16S genes)
3. Train Model B (canonical k-mer Random Forest + fragment augmentation)
4. Extract genome-resolved 16S from the wastewater MAGs
5. Run inference (identify then classify then fuse)
6. **Validate the model against ground truth** (stratified + leave-one-genus-out CV)
7. k-mer feature importance (what drives Model B)
8. Community composition, diversity & phylogeny
9. **Genome-resolved bridge** (pangenome + GO enrichment on flagged pathogens)
10. Summary of outputs


## Objective 0 — Environment & configuration

All paths live here so every downstream cell is reproducible. Names mirror the MAG
repo (`saved_model/` then `saved_model_16s/`, `pathogen_predict.py`, etc.). Edit the
`ROOT` and dataset paths to match your HPC layout, then run top-to-bottom.

In [ ]:
import os, json, subprocess, sys
from pathlib import Path

# ---- repo + data roots (this HPC layout) ---------------------------------
ROOT      = Path(os.environ.get("ROOT", Path.home()/"pathogen-projects"/
                 "complete-chromosome-pathogen-non-pathogen-genomes")).expanduser()
DATASETS  = Path(os.environ.get("DATASETS", Path.home()/"datasets")).expanduser()

# ---- dataset folders (as they exist under ~/datasets) --------------------
DS = {
    "figshare_5cities": DATASETS/"01_figshare_5cities_becsei",           # external 16S (Becsei et al.)
    "ena_assemblies":   DATASETS/"02_ena_PRJEB68319_assemblies",
    "resistome":        DATASETS/"03_msystems_resistome",
    "wastewater_mags":  DATASETS/"04_ncbi_wastewater_metagenome_527639", # 5,328 MAGs
    "extracted_16s":    DATASETS/"05_16s_from_mags",                     # all_16s.fna (this study)
    "pathogen_calls":   DATASETS/"06_16s_pathogen_calls",               # inference output
    "timeseries":       DATASETS/"06_pubmed39215001_timeseries",
}

# ---- primary inputs -------------------------------------------------------
MAGS_DIR      = DS["wastewater_mags"]/"mags_input"          # 5,328 MAG .fna
EXTRACTED_16S = DS["extracted_16s"]/"all_16s.fna"           # genome-resolved rep 16S (~1,321)
PRED_DIR      = DS["pathogen_calls"]                        # *_16s_taxa.csv, *_report.json
FIGSHARE_16S  = DS["figshare_5cities"]                      # optional external validation input

# ---- model artifacts (in the repo) ---------------------------------------
SAVED         = ROOT/"saved_model_16s"
KMER_MODEL    = SAVED/"kmer16s_model.pkl"                   # Model B Random Forest (pre-trained)
CATALOG       = SAVED/"pathogen_16s_catalog.json"          # fusion catalog
PATHOGEN_16S  = SAVED/"genome_16s_pathogen.fasta"          # 826 pathogen 16S (Model B labels)
NONPATH_16S   = SAVED/"genome_16s_nonpathogen.fasta"       # 606 non-pathogen 16S
LABELED_16S   = SAVED/"labeled_16s_full.fasta"             # merged + tagged (build_labeled_16s.py)
REF_FASTA     = ROOT/"ref_16s"/"ncbi_16s_refseq.fasta"     # Model A reference (~26,877)

# ---- outputs --------------------------------------------------------------
OUT        = DATASETS/"16s_pipeline_outputs"
VALID_DIR  = ROOT/"model_b_validation"                     # validate_16s_model.py output
BRIDGE_DIR = OUT/"genome_resolved_bridge"
for d in (OUT, BRIDGE_DIR): d.mkdir(parents=True, exist_ok=True)

# ---- knobs ----------------------------------------------------------------
KMER_K   = 6         # --kmer-k ; canonical collapse -> (4^k + 4^(k/2))/2 = 2,080 features
MIN_LEN  = 400       # min 16S length (bp)
THREADS  = int(os.environ.get("THREADS", "8"))

def sh(cmd, **kw):
    print("[$]", cmd)
    return subprocess.run(cmd, shell=True, check=True, text=True, **kw)

print("ROOT     :", ROOT)
print("DATASETS :", DATASETS)
print("python   :", sys.version.split()[0])


**Software** (conda; mirrors the MAG repo, minus assembly-only tools):

```bash
conda create -n gomics_16s python=3.11 -y && conda activate gomics_16s
pip install pandas numpy scikit-learn matplotlib seaborn scipy statsmodels \
            goatools requests tqdm biopython
conda install -c bioconda blast barrnap ppanggolin mafft fasttree -y
```
`blast` + `barrnap` cover the core 16S workflow; `ppanggolin`, `mafft`, `fasttree`
are used by the downstream phylogeny (Obj. 8) and genome-resolved bridge (Obj. 9).

## Datasets

All inputs live under `~/datasets/`. The pipeline's primary input is the wastewater
MAG catalog (04); its extracted 16S (05) and pathogen calls (06) are produced by this
notebook. The figshare 5-city set (01) is available as an external 16S validation input.

| Folder | Contents | Role |
|--------|----------|------|
| `01_figshare_5cities_becsei` | Becsei et al. 5-city wastewater 16S | external validation (optional) |
| `02_ena_PRJEB68319_assemblies` | ENA assemblies | reference / comparison |
| `03_msystems_resistome` | mSystems resistome data | resistance context |
| `04_ncbi_wastewater_metagenome_527639` | 5,328 wastewater MAGs | **primary input** |
| `05_16s_from_mags` | `all_16s.fna` (~1,321 representative 16S) | genome-resolved 16S |
| `06_16s_pathogen_calls` | `*_16s_taxa.csv`, `*_report.json` | inference output |
| `06_pubmed39215001_timeseries` | longitudinal series | time-series context |


In [ ]:
# confirm the dataset + model paths resolve on this machine
for name, path in DS.items():
    print(f"  [{'OK' if path.exists() else '--'}] {name:18s} {path}")
print()
for label, p in [("MAGs", MAGS_DIR), ("extracted 16S", EXTRACTED_16S),
                 ("Model B model", KMER_MODEL), ("pathogen 16S", PATHOGEN_16S),
                 ("non-path 16S", NONPATH_16S), ("Model A ref", REF_FASTA)]:
    print(f"  [{'OK' if Path(p).exists() else '--'}] {label:16s} {p}")


## Objective 1 — Build the Model A reference (16S RefSeq then BLAST DB)

Model A is approximate nearest-neighbour search over **26,877 taxonomically labeled
16S sequences** (NCBI 16S RefSeq / Targeted Loci). Download the reference and build
the BLAST database. Analogous to the MAG repo's `fix_build_blast_db.py`.

In [ ]:
# ref_16s_download.py + build_16s_blast_db.py (inlined)
REF_FASTA = SAVED/"ref_16s"/"16S_ribosomal_RNA.fasta"
REF_FASTA.parent.mkdir(parents=True, exist_ok=True)

if not (Path(str(REF_DB)+".nsq").exists() or (REF_FASTA.parent/"16S_ribosomal_RNA.nsq").exists()):
    # NCBI prebuilt 16S rRNA BLAST DB (RefSeq Targeted Loci)
    sh(f"cd {REF_FASTA.parent} && "
       f"wget -q -N https://ftp.ncbi.nlm.nih.gov/blast/db/16S_ribosomal_RNA.tar.gz && "
       f"tar -xzf 16S_ribosomal_RNA.tar.gz")
    # If you instead have a raw reference FASTA, build the DB explicitly:
    # sh(f"makeblastdb -in {REF_FASTA} -dbtype nucl -parse_seqids -out {REF_DB}")
    print("Model A reference ready")
else:
    print("Model A DB already present")

# sanity: report the reference size (expect ~26,877)
try:
    info = subprocess.run(f"blastdbcmd -db {REF_FASTA.parent/'16S_ribosomal_RNA'} -info",
                          shell=True, text=True, capture_output=True).stdout
    print(info.splitlines()[0] if info else "run blastdbcmd -info to confirm ~26,877 seqs")
except Exception as e:
    print("info:", e)


## Objective 2 — Build the Model B labeled set

Model B's labels are the **16S genes pulled from the same labeled genomes** used by the
whole-genome classifier, already extracted into two FASTAs in `saved_model_16s/`:
`genome_16s_pathogen.fasta` (**826**) and `genome_16s_nonpathogen.fasta` (**606**).
Their headers carry only a genome accession (e.g. `044324975.1_16S_3`), so here we merge
them into one labeled file, tag each record `id|label|genus`, and resolve the genus from
the genome-list JSONs. The genus is what the leave-one-genus-out validation groups on
(and, because 16S is multi-copy, it also keeps a genome's copies in the same fold).

In [ ]:
# build_labeled_16s.py (inlined): merge + tag the two training FASTAs
import re
ACC = re.compile(r'(GC[AF]_)?(\d{6,}\.\d+)')
ORG = re.compile(r'\b([A-Z][a-z]{2,})\s+([a-z]{2,})')
STOP = {"Complete","Whole","Genome","Assembly","Chromosome","Scaffold","Contig","Strain"}
def core(s):
    m = ACC.search(s or ""); return m.group(2) if m else None

# accession-core -> genus, mined from the genome-list JSONs (if present)
acc2genus = {}
for jf in ["pathogen_complete_genomes_fixed.json",
           "non_pathogen_complete_genomes_fixed.json",
           "complete_pathogen_genome_only.json"]:
    p = ROOT/jf
    if not p.exists(): continue
    try: data = json.load(open(p))
    except Exception: continue
    for r in (data.values() if isinstance(data, dict) else data):
        blob = " ".join(str(v) for v in r.values()) if isinstance(r, dict) else str(r)
        c = core(blob)
        for g, sp in ORG.findall(blob):
            if g not in STOP:
                if c: acc2genus.setdefault(c, g)
                break

def hdr_core(h):
    return core(h.split("_16S_")[0].lstrip(">")) or h.split("_16S_")[0].lstrip(">")

def read_fa(p):
    name=None; buf=[]
    for ln in open(p):
        if ln.startswith(">"):
            if name: yield name, "".join(buf)
            name=ln[1:].strip(); buf=[]
        else: buf.append(ln.strip())
    if name: yield name, "".join(buf)

cov = tot = 0
with open(LABELED_16S, "w") as out:
    for src, label in [(PATHOGEN_16S, "pathogen"), (NONPATH_16S, "non_pathogen")]:
        for name, seq in read_fa(src):
            genus = acc2genus.get(hdr_core(name))
            grp = genus if genus else hdr_core(name)          # fall back to genome accession
            cov += 1 if genus else 0; tot += 1
            out.write(f">{name}|{label}|{grp}\n{seq}\n")
print(f"labeled set: {tot} genes | genus-resolved {cov} ({100*cov//max(tot,1)}%) -> {LABELED_16S}")


## Objective 3 — Train Model B (canonical k-mer Random Forest)

Mirror of the MAG repo's `pathogen_predict.py setup` / `pathogenicity_ml.py`, but the
feature space is the **2,080-dim canonical 6-mer frequency vector** rather than a
705-dim gene-family vector. Class-weight-balanced Random Forest; **fragment
augmentation** adds random sub-windows so the model is robust to short amplicon reads.

In [ ]:
# train_16s_model.py (inlined)
import numpy as np, itertools, random, pickle
random.seed(42); np.random.seed(42)
COMP = str.maketrans("ACGT", "TGCA"); rc = lambda s: s.translate(COMP)[::-1]

def canonical_index(k):
    keys = sorted({min(km, rc(km)) for km in map("".join, itertools.product("ACGT", repeat=k))})
    return {km: i for i, km in enumerate(keys)}

def vec(seq, k, idx):
    seq = seq.upper(); v = np.zeros(len(idx), np.float32); n = 0
    for i in range(len(seq)-k+1):
        km = seq[i:i+k]
        if any(c not in "ACGT" for c in km): continue
        j = idx.get(min(km, rc(km)))
        if j is not None: v[j] += 1; n += 1
    return v/n if n else v

def read_fa(p):
    name=None; buf=[]
    for ln in open(p):
        if ln.startswith(">"):
            if name: yield name, "".join(buf)
            name=ln[1:].strip(); buf=[]
        else: buf.append(ln.strip())
    if name: yield name, "".join(buf)

def frags(seq, n, lo=250, hi=500):
    out=[]; L=len(seq)
    for _ in range(n):
        if L<=lo: break
        wl=random.randint(lo, min(hi,L)); s=random.randint(0, L-wl); out.append(seq[s:s+wl])
    return out

idx = canonical_index(KMER_K); X=[]; y=[]; LAB={"pathogen":1,"non_pathogen":0}
for hdr, seq in read_fa(LABELED_16S):
    lab = LAB[hdr.split("|")[1]]
    X.append(vec(seq,KMER_K,idx)); y.append(lab)
    for fr in frags(seq,3): X.append(vec(fr,KMER_K,idx)); y.append(lab)
X=np.vstack(X); y=np.array(y)
print("training matrix:", X.shape, "| positives:", int(y.sum()))

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=400, class_weight="balanced",
                            random_state=42, n_jobs=-1).fit(X, y)
pickle.dump(rf, open(KMER_MODEL, "wb"))
json.dump({str(v):k for k,v in idx.items()}, open(KMER_INDEX, "w"))
print("saved Model B ->", KMER_MODEL, "|", len(idx), "canonical features")


## Objective 4 — Extract genome-resolved 16S from the wastewater MAGs

The application input. `extract_16s.py` runs `barrnap` on all 5,328 MAGs, keeps one
`>{accession}__16S` representative per genome (>=400 bp, longest), and preserves the
link from each 16S sequence to its source MAG. This is the genome-resolved substrate
for the whole study.

In [ ]:
# extract_16s.py  (the script committed in the repo)
if not Path(EXTRACTED_16S).exists():
    sh(f"MAGS={MAGS_DIR} OUT16S={EXTRACTED_16S.parent} "
       f"WORKERS={THREADS} python {ROOT/'extract_16s.py'}")
n = subprocess.run(f"grep -c '^>' {EXTRACTED_16S}", shell=True, text=True,
                   capture_output=True).stdout.strip()
print(f"genome-resolved 16S records: {n}   (expected ~1,321 of 5,328 MAGs = 24.8%)")


## Objective 5 — Run inference (identify then classify then fuse)

Mirror of the MAG repo's `pathogen_predict.py` inference. The `16s` subcommand runs
Model A (BLAST identity then taxonomy), Model B (k-mer RF then P(pathogen)), and the
precision-biased fusion layer, emitting a per-taxon report and CSV.

In [ ]:
# pathogen_predict.py 16s  (unified inference tool, 16S subcommand)
sh(f"cd {ROOT} && python pathogen_predict.py 16s "
   f"--input {EXTRACTED_16S} --output {PRED_DIR}")

import pandas as pd, glob
taxa = pd.read_csv(sorted(glob.glob(str(PRED_DIR/'*_16s_taxa.csv')))[0])
print("reported taxa:", len(taxa))
print(taxa['final_call'].value_counts().to_dict())
taxa.head()


## Objective 6 — Validate Model B against ground truth (star)

**This is the accuracy the paper reports.** `validate_16s_model.py` evaluates Model B
on the 1,432 labeled genes under two schemes:

- **Stratified k-fold** (in-distribution performance).
- **Leave-one-genus-out** removes every test genus from training, so a correct call
  cannot come from having seen that genus. This is the leakage-controlled
  generalization number to cite as *the model's accuracy*.

Outputs: `metrics_summary.json`, per-fold CSVs, ROC / confusion / calibration figures.

In [ ]:
sh(f"cd {ROOT} && PYTHONWARNINGS=ignore python validate_16s_model.py "
   f"--labeled-fasta {LABELED_16S} --kmer-k {KMER_K} --folds 5 --augment 2 "
   f"--n-estimators 250 --min-genus-size 2 --outdir {VALID_DIR}")

metrics = json.load(open(VALID_DIR/"metrics_summary.json"))
import pandas as pd
tab = pd.DataFrame({
    "stratified_kfold": metrics["stratified_kfold"],
    "leave_genus_out":  metrics["leave_genus_out"],
}).loc[["accuracy","balanced_accuracy","sensitivity_recall","specificity",
        "precision","f1","mcc","auroc","auprc"]].round(3)
print("\n==> report the leave_genus_out column as the paper's accuracy\n")
print(tab.to_string())
tab.to_csv(VALID_DIR/"performance_table_for_paper.csv")


In [ ]:
# show the figures inline
from IPython.display import Image, display
for f in ["fig_roc.png","fig_confusion.png","fig_calibration.png"]:
    p = VALID_DIR/f
    if p.exists(): display(Image(str(p)))


## Objective 7 — k-mer feature importance (what drives Model B)

The 16S analog of the MAG repo's `feature_importance_genes_go.py` Step 1: which
canonical 6-mers most drive the pathogen decision. (The gene-to-GO steps do not apply
to a single gene; the genomic interpretation is provided by the bridge in Objective 9.)

In [ ]:
import pickle, json, pandas as pd
rf   = pickle.load(open(KMER_MODEL, "rb"))
idx  = json.load(open(KMER_INDEX))                       # {index: kmer}
kmers = [idx[str(i)] for i in range(len(rf.feature_importances_))]
imp = pd.DataFrame({"kmer": kmers, "importance": rf.feature_importances_})
imp = imp.sort_values("importance", ascending=False).head(25).reset_index(drop=True)
imp.to_csv(OUT/"top_kmers_model_b.csv", index=False)

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(6,5))
plt.barh(imp["kmer"][::-1], imp["importance"][::-1], color="#2f6ea5")
plt.xlabel("Random-forest importance"); plt.title("Top canonical 6-mers driving Model B")
plt.tight_layout(); plt.savefig(OUT/"top_kmers_model_b.png", dpi=200)
print(imp.head(10).to_string(index=False))


## Objective 8 — Community composition, diversity & phylogeny

Downstream ecological analysis of the reported taxa: abundance, alpha diversity
(Shannon), and a FastTree phylogeny of the representative 16S sequences
(MAFFT alignment then FastTree).

In [ ]:
import pandas as pd, glob, numpy as np
taxa = pd.read_csv(sorted(glob.glob(str(PRED_DIR/'*_16s_taxa.csv')))[0])

g = taxa.groupby("genus")["num_reads"].sum().sort_values(ascending=False)
p = g/g.sum(); shannon = float(-(p*np.log(p)).sum())
print("distinct genera:", len(g), "| Shannon H':", round(shannon,3))
print(g.head(15).to_string())

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(6,4.5)); plt.barh(g.head(15).index[::-1], g.head(15).values[::-1], color="#3b6ea5")
plt.xlabel("representative 16S reads"); plt.title("Top genera")
plt.tight_layout(); plt.savefig(OUT/"top_genera.png", dpi=200)


In [ ]:
# phylogeny of representative 16S (MAFFT -> FastTree). Skips gracefully if tools absent.
aln = OUT/"rep16s.aln"; tree = OUT/"rep16s.nwk"
try:
    sh(f"mafft --auto --thread {THREADS} {EXTRACTED_16S} > {aln}")
    sh(f"FastTree -nt -gtr {aln} > {tree}")
    print("tree written:", tree)
except Exception as e:
    print("phylogeny skipped (install mafft + fasttree):", e)


## Objective 9 — Genome-resolved bridge: pangenome + GO enrichment (star)

This is where the 16S calls reconnect to genomes. For every taxon flagged
**PATHOGEN / OPPORTUNIST**, we take its **source MAG(s)** and run the *same*
downstream the MAG repo uses: a `ppanggolin` pangenome and GO-term enrichment
(`feature_importance_genes_go.py` then `go_pathway_prediction.py`). This yields the
gene families and biological pathways behind each flagged organism, the genomic
evidence a bare 16S label cannot provide.

In [ ]:
# 9a. collect source MAGs of flagged (pathogen/opportunist) taxa
import pandas as pd, glob, shutil
taxa = pd.read_csv(sorted(glob.glob(str(PRED_DIR/'*_16s_taxa.csv')))[0])
flagged = taxa[taxa["final_call"].isin(["PATHOGEN","OPPORTUNIST"])].copy()

flagged_dir = BRIDGE_DIR/"flagged_source_mags"; flagged_dir.mkdir(parents=True, exist_ok=True)
def acc_of(row):  # if your taxa CSV carries the source accession, use it directly
    return row.get("source_mag") or row.get("otu_id")
copied = []
for _, r in flagged.iterrows():
    acc = str(acc_of(r)).split("__")[0]
    src = Path(MAGS_DIR)/f"{acc}.fna"
    if src.exists():
        shutil.copy(src, flagged_dir/src.name); copied.append(acc)
print(f"flagged taxa: {len(flagged)} | source MAGs gathered: {len(copied)} -> {flagged_dir}")


In [ ]:
# 9b. pangenome of the flagged genomes (ppanggolin) — mirrors pangenome_code.py
pg_out = BRIDGE_DIR/"flagged_pangenome"
try:
    sh("ls " + str(flagged_dir) + "/*.fna | awk '{n=split($0,a,\"/\"); f=a[n]; "
       "sub(/\\.fna$/,\"\",f); print f\"\\t\"$0}' > " + str(BRIDGE_DIR/"flagged_list.tsv"))
    sh(f"ppanggolin workflow --anno_or_fasta {BRIDGE_DIR}/flagged_list.tsv "
       f"--output {pg_out} --cpu {THREADS} --force")
    print("pangenome ->", pg_out)
except Exception as e:
    print("ppanggolin step skipped (install ppanggolin / needs >=2 genomes):", e)


In [ ]:
# 9c. GO enrichment on the flagged pangenome — reuse the MAG repo scripts unchanged
#     (they read .gbff + the ppanggolin matrix; point their CONFIG at the bridge dir).
try:
    sh(f"cd {ROOT} && python feature_importance_genes_go.py")   # Steps 1-7 -> genes + GO terms
    sh(f"cd {ROOT} && python go_pathway_prediction.py")         # Steps 1-6 -> enrichment + plots
    print("GO enrichment complete -> see go_pathway_summary.csv + *.png")
except Exception as e:
    print("GO bridge skipped (set CONFIG paths in the two scripts to the bridge dir):", e)


## Objective 10 — Summary of outputs

| Objective | Key output |
|---|---|
| 1 Model A reference | `saved_model_16s/ref_16s/` BLAST DB (~26,877 seqs) |
| 2 Model B labels | `saved_model_16s/labeled_16s.fasta` (826 + 606) |
| 3 Model B train | `saved_model_16s/kmer16s_model.pkl`, `kmer_index.json` |
| 4 16S extraction | `05_16s_from_mags/all_16s.fna` (~1,321) |
| 5 Inference | `06_16s_pathogen_calls/*_16s_taxa.csv`, `*_report.json` |
| 6 **Validation** | `model_b_validation/metrics_summary.json` + figures (paper accuracy) |
| 7 Feature importance | `top_kmers_model_b.csv/.png` |
| 8 Diversity/phylogeny | `top_genera.png`, `rep16s.nwk` |
| 9 **Genome-resolved bridge** | `genome_resolved_bridge/` pangenome + GO enrichment |


In [ ]:
print("Pipeline complete. Headline numbers:")
try:
    m = json.load(open(VALID_DIR/"metrics_summary.json"))
    lgo = m["leave_genus_out"]
    print(f"  Model B generalization (leave-one-genus-out): "
          f"acc={lgo['accuracy']:.3f}  AUROC={lgo['auroc']:.3f}  "
          f"sens={lgo['sensitivity_recall']:.3f}  spec={lgo['specificity']:.3f}")
except Exception as e:
    print("  run Objective 6 to produce the accuracy numbers:", e)
